# Week 4: One Hotel Payment, Retry and Replan

A 120-minute lab: setup 20 min, teacher execution walkthrough 20 min, student tasks 45 min, verification 20 min, discussion 15 min.

Exactly three tasks in `src/travel_challenge/week4/payment_execution.py`: `previous_receipt` (15 min), `choose_action` (10 min), `reauthorize` (20 min).

Two fixed scenarios only: a lost technical response after payment, and a hotel price increase before payment. The teacher supplies the execution skeleton. Use an in-memory receipt dictionary; no wallet, ledger, database or transaction platform.

Reuse the exported W1/W2 plan and Week 3 official AP2 models/local authorization. All hotel payments are simulated; no reservation or real payment occurs.

## Setup and Week 3 Prerequisite (20 min)

Complete the Week 3 source tasks first. Read the same exported trip plan; construct official mandates and explicitly approve the selected hotel using Week 3 functions.

Authorization is a detached local purchase snapshot, not a production user credential. Execution pays exactly that snapshot's conditions. The receipt dictionary exists only in this kernel session.

In [ ]:
import json
import sys
import importlib
import subprocess
from pathlib import Path
from travel_challenge.week3 import authorization
from travel_challenge.week1 import trip_planner
from travel_challenge.week4 import payment_execution as execution

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
plan_path = ROOT / 'trip_plan.json'
assert plan_path.exists(), 'Export your Week 1/2 plan before continuing.'
plan = json.loads(plan_path.read_text(encoding='utf-8'))
purchase = authorization.hotel_purchase(plan)
checkout, payment = authorization.make_mandates(purchase)
authorized_purchase = authorization.approve_purchase(purchase, checkout, payment, consent=True)
receipts = {}
print('Running this approval cell explicitly approves this simulated hotel purchase:', authorized_purchase)

## Teacher Execution Walkthrough (20 min)

`run_scenario` accepts the original plan and approved purchase. `execute_hotel` binds execution to that exact purchase and uses the student receipt lookup.

Technical flow: payment recorded -> response lost -> RETRY with the same purchase and payment ID -> previous receipt.

Price-change flow: no payment -> new simulated hotel quote -> W1 `make_plan` updates hotel, total and budget -> block if over budget -> otherwise fresh W3 mandates and explicit approval -> new payment ID.

Only three student tasks remain. Budget recalculation, execution and process control are instructor-provided. All receipts are simulated and local to this kernel.

In [ ]:
print('Trip request:', plan['request'])
print('Selected flight:', plan['flight'])
print('Hotel purchase:', purchase)
print('Trip total SGD:', plan['total_cents'] / 100)
print('Budget SGD:', plan['request']['budget_cents'] / 100, 'Within budget:', plan['within_budget'])

## Task 1: Duplicate-Payment Prevention (15 min)

Implement `week4/payment_execution.py::previous_receipt`:

- New payment ID: return `None`.
- Existing ID with the identical complete purchase: return a detached copy of the previous receipt.
- Existing ID with conflicting purchase details: raise `ValueError`.

Compare amount, currency, merchant and quote identity by comparing the complete purchase dictionary. Do not create a new payment ID during a technical retry. Do not add a wallet balance or ledger.

In [ ]:
importlib.reload(execution)
check = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_week4.py', '-m', 'student', '-k', 'previous_receipt', '-q'], cwd=ROOT, capture_output=True, text=True)
print(check.stdout)
assert check.returncode == 0, check.stderr

## Task 2: RETRY or REPLAN (10 min)

Implement `week4/payment_execution.py::choose_action`: `TECHNICAL_ERROR` returns `RETRY`, `HOTEL_PRICE_CHANGED` returns `REPLAN`, and any unknown reason raises `ValueError`.

The teacher skeleton controls execution and IDs. Your function makes only this small decision; do not introduce a planning engine or another specialist.

## Scenario 1: Technical Retry

The simulated hotel records a successful payment, but its response is lost. The coordinator receives `TECHNICAL_ERROR` and retries exactly the same purchase with the same payment ID and authorization.

The second call must return the stored receipt, not create another payment. The receipt dictionary must contain exactly one entry.

In [ ]:
importlib.reload(execution)
retry_receipts = {}
retry_result = execution.run_scenario(plan, authorized_purchase, 'TECHNICAL_ERROR', retry_receipts)
assert retry_result['action'] == 'RETRY'
assert retry_result['plan'] == plan
assert retry_result['receipt']['payment_id'] == retry_result['original_payment_id']
assert len(retry_receipts) == 1
again = execution.execute_hotel(retry_result['original_payment_id'], purchase, authorized_purchase, retry_receipts)
assert again == retry_result['receipt']
print('Same ID, same receipt, unchanged plan, one simulated payment:', retry_result)

## Task 3: Reauthorize the Changed Hotel (20 min)

Implement `week4/payment_execution.py::reauthorize`. Use Week 3 `make_mandates` for the changed purchase and `approve_purchase` with the supplied explicit consent. Return the new authorized snapshot.

Do not return the old snapshot, reuse an old mandate, silently approve, or mutate the original purchase. Without consent, approval must fail before payment.

## Scenario 2: Hotel Price Change and Budget

Use the request, budget and hotel purchase saved in the W1/W2 exported plan. If `trip_scenario.py` changes, regenerate that plan through W1/W2; do not replace its budget or prices with current scenario values.

The first simulated feedback adds SGD 50.00 to the full-stay hotel quote and gives it a new simulated quote ID. Keep the flight and reuse W1 `make_plan` to recalculate the trip total and budget status. Old authorization must fail. Over-budget results stop before new authorization or payment; within-budget results require fresh mandates, explicit consent and a new payment ID.

For the verified default SGD 6,400.22 trip, the small increase gives SGD 6,450.22, within its saved budget. A separate SGD 700.00 increase gives SGD 7,100.22 and is blocked. If an edited scenario has a larger saved budget, the supplied demonstration increases this second amount just enough to exceed it. The budget itself never changes.

Both price changes are instructor simulations, not live hotel repricing or another student task.

In [ ]:
importlib.reload(execution)
price_receipts = {}
feedback = execution.execute_hotel('price-change-demo', purchase, authorized_purchase, price_receipts, feedback='HOTEL_PRICE_CHANGED')
changed_purchase = feedback['purchase']
assert not price_receipts
try:
    execution.execute_hotel('must-be-blocked', changed_purchase, authorized_purchase, price_receipts)
except PermissionError as error:
    print('Old authorization rejected:', error)
else:
    raise AssertionError('Changed purchase used old authorization')
print('New simulated hotel quote:', changed_purchase)

In [ ]:
importlib.reload(execution)
print('Explicitly approving the displayed changed purchase only if the updated trip is within budget:', changed_purchase)
replan_result = execution.run_scenario(plan, authorized_purchase, 'HOTEL_PRICE_CHANGED', price_receipts, consent_after_change=True)
updated_plan = replan_result['plan']
assert replan_result['action'] == 'REPLAN'
assert updated_plan['total_cents'] == plan['total_cents'] + 5000
assert updated_plan['hotel']['id'] == changed_purchase['quote_id']
print('Updated total SGD:', updated_plan['total_cents'] / 100, 'Within budget:', updated_plan['within_budget'])
if not updated_plan['within_budget']:
    assert replan_result['reason'] == 'OVER_BUDGET'
    assert replan_result['receipt'] is None and not price_receipts
    print('Over budget: no new authorization or payment.')
else:
    assert replan_result['receipt']['payment_id'] != replan_result['original_payment_id']
    assert replan_result['receipt']['purchase'] == changed_purchase
    assert len(price_receipts) == 1
    print('Fresh authorization and new payment ID:', replan_result['receipt'])

over_increase_cents = 70000
if plan['total_cents'] + over_increase_cents <= plan['request']['budget_cents']:
    over_increase_cents = plan['request']['budget_cents'] - plan['total_cents'] + 1
over_hotel = {**plan['hotel'], 'id': plan['hotel']['id'] + '-SIMULATED-REPRICE',
              'amount_cents': plan['hotel']['amount_cents'] + over_increase_cents}
over_plan = trip_planner.make_plan(plan['request'], plan['flight'], over_hotel)
assert over_plan['total_cents'] == plan['total_cents'] + over_increase_cents
assert not over_plan['within_budget']
print('Larger simulated hotel increase SGD:', over_increase_cents / 100)
print('Trip total SGD:', over_plan['total_cents'] / 100, 'Saved budget SGD:', plan['request']['budget_cents'] / 100)
try:
    authorization.hotel_purchase(over_plan)
except ValueError:
    print('Over the saved budget: blocked before authorization or payment execution.')
else:
    raise AssertionError('An over-budget plan reached hotel authorization')

## Verification (20 min)

Teacher checks: `python -m pytest -m scaffold -q`.

Week 4 student acceptance: `python -m pytest tests/test_week4.py -m student -q`. Full project: `python -m pytest -q`.

Student checks remain failing until the source TODOs are completed. No skip/xfail hides them. The two scenario tests require the Week 3 authorization tasks too. Unit-test supplier examples are test-only; the notebook uses your exported plan.

In [ ]:
check = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_week4.py', '-q'], cwd=ROOT, capture_output=True, text=True)
print(check.stdout)
assert check.returncode == 0, 'Complete the three Week 4 tasks and Week 3 prerequisites. ' + check.stderr

## Four-Week Debrief (15 min)

Explain the complete sequence: travel request -> W1 REST plan -> W2 official A2A/MCP plan -> W3 hotel mandates and approval -> W4 simulated execution -> retry or hotel replan.

Why does a technical retry keep the payment ID? Why does a changed quote need both new approval and a new payment ID? Which fields bind the payment to the purchase?

Weather x402 is a separate small paid-resource example, not another payment rail for the hotel.

## Deliberate Limits

Receipts disappear when the kernel exits. This prevents duplicates only inside the current teaching session; it is not durable, concurrent or production payment infrastructure.

The price increase and new quote ID are explicitly simulated. No real booking, bank/card payment, USDC settlement, wallet balance or ledger is involved. The local authorization snapshot is trusted teaching data, not a full AP2 cryptographic credential.

The final architecture consists only of real REST, the required official protocol plumbing, these simple authorization/execution functions and four weekly test files.